In [1]:
import pandas as pd # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None # Menonaktifkan peringatan chaining
import numpy as np # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed) # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt # Matplotlib untuk visualisasi data
import seaborn as sns # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score

import datetime as dt # Manipulasi data waktu dan tanggal
import re # Modul untuk bekerja dengan ekspresi reguler
import string # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize # Tokenisasi teks
from nltk.corpus import stopwords # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt_tab') # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords') # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.6 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
app_reviews_df = pd.read_csv('ulasan_aplikasi.csv') # Membaca file CSV dari URL

In [3]:
# Menampilkan lima baris pertama dari DataFrame app_reviews_df
app_reviews_df.head()

,Review
0,"Minta tolong diperbaiki untuk masalah login, s..."
1,Saat sedang nonton Piala Dunia setiap beberapa...
2,Rencananya mau beli paket MaxStream buat nonto...
3,Ini aplikasi paling membosankan dan paling men...
4,"sangat tidak nyaman, sesi expired sangat serin..."


In [4]:
# Menampilkan informasi tentang DataFrame app_reviews_df
app_reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69594 entries, 0 to 69593
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  69594 non-null  object
dtypes: object(1)
memory usage: 543.8+ KB


In [5]:
# Membuat DataFrame baru (clean_df) dengan menghapus baris yang memiliki nilai yang hilang (NaN) dari app_reviews_df
clean_df = app_reviews_df.dropna()

In [6]:
# Menghapus baris duplikat dari DataFrame clean_df
clean_df = clean_df.drop_duplicates()

# Menghitung jumlah baris dan kolom dalam DataFrame clean_df setelah menghapus duplikat
jumlah_ulasan_setelah_hapus_duplikat, jumlah_kolom_setelah_hapus_duplikat = clean_df.shape

In [7]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # Memecah teks menjadi daftar kata
    words = text.split()

    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]

    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [8]:
slangwords = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal"}
def fix_slangwords(text):
     words = text.split()
     fixed_words = []

     for word in words:
       if word.lower() in slangwords:
           fixed_words.append(slangwords[word.lower()])
       else:
           fixed_words.append(word)

     fixed_text = ' '.join(fixed_words)
     return fixed_text

In [9]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['Review'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

In [10]:
clean_df.head()

,Review,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir
0,"Minta tolong diperbaiki untuk masalah login, s...",Minta tolong diperbaiki untuk masalah login sa...,minta tolong diperbaiki untuk masalah login sa...,minta tolong diperbaiki untuk masalah login sa...,"[minta, tolong, diperbaiki, untuk, masalah, lo...","[tolong, diperbaiki, login, login, akun, menit...",tolong diperbaiki login login akun menit habis...
1,Saat sedang nonton Piala Dunia setiap beberapa...,Saat sedang nonton Piala Dunia setiap beberapa...,saat sedang nonton piala dunia setiap beberapa...,saat sedang nonton piala dunia setiap beberapa...,"[saat, sedang, nonton, piala, dunia, setiap, b...","[nonton, piala, dunia, menit, muncul, error, b...",nonton piala dunia menit muncul error berulang...
2,Rencananya mau beli paket MaxStream buat nonto...,Rencananya mau beli paket MaxStream buat nonto...,rencananya mau beli paket maxstream buat nonto...,rencananya mau beli paket maxstream buat nonto...,"[rencananya, mau, beli, paket, maxstream, buat...","[rencananya, beli, paket, maxstream, nonton, p...",rencananya beli paket maxstream nonton piala d...
3,Ini aplikasi paling membosankan dan paling men...,Ini aplikasi paling membosankan dan paling men...,ini aplikasi paling membosankan dan paling men...,ini aplikasi paling membosankan dan paling men...,"[ini, aplikasi, paling, membosankan, dan, pali...","[aplikasi, membosankan, menyusahkan, aplikasi,...",aplikasi membosankan menyusahkan aplikasi apap...
4,"sangat tidak nyaman, sesi expired sangat serin...",sangat tidak nyaman sesi expired sangat sering...,sangat tidak nyaman sesi expired sangat sering...,sangat tidak nyaman sesi expired sangat sering...,"[sangat, tidak, nyaman, sesi, expired, sangat,...","[nyaman, sesi, expired, jaringan, indihome, te...",nyaman sesi expired jaringan indihome tertera ...


In [11]:
import csv
import requests
from io import StringIO

# Membaca data kamus kata-kata positif dari GitHub
lexicon_positive = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub

if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma

    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_positive[row[0]] = int(row[1])
        # Menambahkan kata-kata positif dan skornya ke dalam kamus lexicon_positive
else:
    print("Failed to fetch positive lexicon data")

# Membaca data kamus kata-kata negatif dari GitHub
lexicon_negative = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub

if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma

    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_negative[row[0]] = int(row[1])
        # Menambahkan kata-kata negatif dan skornya dalam kamus lexicon_negative
else:
    print("Failed to fetch negative lexicon data")

In [12]:
# Fungsi untuk menentukan polaritas sentimen dari tweet

def sentiment_analysis_lexicon_indonesia(text):
    #for word in text:

    score = 0
    # Inisialisasi skor sentimen ke 0

    for word in text:
        # Mengulangi setiap kata dalam teks

        if (word in lexicon_positive):
            score = score + lexicon_positive[word]
            # Jika kata ada dalam kamus positif, tambahkan skornya ke skor sentimen

    for word in text:
        # Mengulangi setiap kata dalam teks (sekali lagi)

        if (word in lexicon_negative):
            score = score + lexicon_negative[word]
            # Jika kata ada dalam kamus negatif, kurangkan skornya dari skor sentimen

    polarity=''
    # Inisialisasi variabel polaritas

    if (score >= 0):
        polarity = 'positive'
        # Jika skor sentimen lebih besar atau sama dengan 0, maka polaritas adalah positif
    elif (score < 0):
        polarity = 'negative'
        # Jika skor sentimen kurang dari 0, maka polaritas adalah negatif

    # else:
    #     polarity = 'neutral'
    # Ini adalah bagian yang bisa digunakan untuk menentukan polaritas netral jika diperlukan

    return score, polarity
    # Mengembalikan skor sentimen dan polaritas teks

In [13]:
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]
print(clean_df['polarity'].value_counts())

polarity
positive    26842
negative    25226
Name: count, dtype: int64


In [14]:
clean_df.head()

,Review,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir,polarity_score,polarity
0,"Minta tolong diperbaiki untuk masalah login, s...",Minta tolong diperbaiki untuk masalah login sa...,minta tolong diperbaiki untuk masalah login sa...,minta tolong diperbaiki untuk masalah login sa...,"[minta, tolong, diperbaiki, untuk, masalah, lo...","[tolong, diperbaiki, login, login, akun, menit...",tolong diperbaiki login login akun menit habis...,10,positive
1,Saat sedang nonton Piala Dunia setiap beberapa...,Saat sedang nonton Piala Dunia setiap beberapa...,saat sedang nonton piala dunia setiap beberapa...,saat sedang nonton piala dunia setiap beberapa...,"[saat, sedang, nonton, piala, dunia, setiap, b...","[nonton, piala, dunia, menit, muncul, error, b...",nonton piala dunia menit muncul error berulang...,-29,negative
2,Rencananya mau beli paket MaxStream buat nonto...,Rencananya mau beli paket MaxStream buat nonto...,rencananya mau beli paket maxstream buat nonto...,rencananya mau beli paket maxstream buat nonto...,"[rencananya, mau, beli, paket, maxstream, buat...","[rencananya, beli, paket, maxstream, nonton, p...",rencananya beli paket maxstream nonton piala d...,7,positive
3,Ini aplikasi paling membosankan dan paling men...,Ini aplikasi paling membosankan dan paling men...,ini aplikasi paling membosankan dan paling men...,ini aplikasi paling membosankan dan paling men...,"[ini, aplikasi, paling, membosankan, dan, pali...","[aplikasi, membosankan, menyusahkan, aplikasi,...",aplikasi membosankan menyusahkan aplikasi apap...,-27,negative
4,"sangat tidak nyaman, sesi expired sangat serin...",sangat tidak nyaman sesi expired sangat sering...,sangat tidak nyaman sesi expired sangat sering...,sangat tidak nyaman sesi expired sangat sering...,"[sangat, tidak, nyaman, sesi, expired, sangat,...","[nyaman, sesi, expired, jaringan, indihome, te...",nyaman sesi expired jaringan indihome tertera ...,10,positive


In [15]:
# Pisahkan data menjadi fitur (text_akhir) dan label (polarity)
X = clean_df['text_akhir']
y = clean_df['polarity']

# Bagi data menjadi data latih dan data uji terlebih dahulu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=200, min_df=17, max_df=0.8 )

# Fit TF-IDF hanya pada data pelatihan dan transformasikan data pelatihan dan pengujian
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


# Kembalikan data yang sudah diubah dengan TF-IDF ke X_train dan X_test supaya konsisten dengan sel-sel berikutnya
X_train = X_train_tfidf
X_test = X_test_tfidf

In [17]:
# Membuat objek model SVM
svm_model = SVC(kernel='linear', random_state=42)

# Melatih model SVM pada data pelatihan
svm_model.fit(X_train, y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_svm = svm_model.predict(X_train)
y_pred_test_svm = svm_model.predict(X_test)

# Evaluasi akurasi model SVM
accuracy_train_svm = accuracy_score(y_pred_train_svm, y_train)
accuracy_test_svm = accuracy_score(y_pred_test_svm, y_test)

# Menampilkan akurasi
print('SVM - accuracy_train:', accuracy_train_svm)
print('SVM - accuracy_test:', accuracy_test_svm)

SVM - accuracy_train: 0.883804676621693
SVM - accuracy_test: 0.8807374687920108


In [18]:
# Membuat objek model Random Forest
random_forest = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())

# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9565227829260095
Random Forest - accuracy_test: 0.8683502976762051


In [19]:
# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression()

# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train.toarray())
y_pred_test_lr = logistic_regression.predict(X_test.toarray())

# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)

# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)

# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Logistic Regression - accuracy_train: 0.8858933115667162
Logistic Regression - accuracy_test: 0.8828500096024582


In [20]:
# Membuat model Decision Tree
dt_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=50,
    random_state=42
)

# Training
dt_model.fit(X_train, y_train)

# Prediksi
y_pred_train_dt = dt_model.predict(X_train)
y_pred_test_dt = dt_model.predict(X_test)

# Evaluasi
accuracy_train_dt = accuracy_score(y_train, y_pred_train_dt)
accuracy_test_dt = accuracy_score(y_test, y_pred_test_dt)

print("Decision Tree - accuracy_train:", accuracy_train_dt)
print("Decision Tree - accuracy_test:", accuracy_test_dt)

Decision Tree - accuracy_train: 0.9245690689969751
Decision Tree - accuracy_test: 0.8513539466103323


In [21]:
# Inference / Testing dengan output kelas kategorikal
category_map = {'positive': 'positif', 'negative': 'negatif', 'neutral': 'netral'}

sample_texts = [
    'Aplikasi Maxstream saat menonton piala dunia sering lemot',
    'Sering terjadi gangguan, koneksi lemot dan aplikasi crash.',
    'Aplikasi berjalan dengan lancar saat menonton'
]

# Pra-pemrosesan teks yang sama seperti pada data pelatihan

def preprocess_text(text):
    cleaned = cleaningText(text)
    lower = casefoldingText(cleaned)
    fixed = fix_slangwords(lower)
    tokens = tokenizingText(fixed)
    filtered = filteringText(tokens)
    return toSentence(filtered)

sample_cleaned = [preprocess_text(text) for text in sample_texts]
sample_features = tfidf.transform(sample_cleaned)

sample_predictions = svm_model.predict(sample_features)
sample_categories = [category_map.get(pred, pred) for pred in sample_predictions]

inference_df = pd.DataFrame({
    'Teks Asli': sample_texts,
    'Teks Bersih': sample_cleaned,
    'Prediksi Kelas': sample_categories
})

inference_df

,Teks Asli,Teks Bersih,Prediksi Kelas
0,Aplikasi Maxstream saat menonton piala dunia s...,aplikasi maxstream menonton piala dunia lemot,negatif
1,"Sering terjadi gangguan, koneksi lemot dan apl...",gangguan koneksi lemot aplikasi crash,negatif
2,Aplikasi berjalan dengan lancar saat menonton,aplikasi berjalan lancar menonton,positif
